In [ ]:
import random
import os
import matplotlib.pyplot as plt
import networkx as nx

In [ ]:
# ============================================================
# CONFIGURACIÓN
# ============================================================

# ------------------------------------------------------------
# Fuente de la matriz
# ------------------------------------------------------------
# "manual"      -> utilizar la matriz escrita aquí
# "archivo"     -> leer la matriz desde un archivo .txt
# "aleatoria"   -> generar una matriz aleatoria

fuente_matriz = "manual"


# ------------------------------------------------------------
# Matriz manual
# ------------------------------------------------------------

matriz = [
    [0, 1, 1, 0, 0],
    [1, 0, 1, 1, 0],
    [1, 1, 0, 1, 1],
    [0, 1, 1, 0, 1],
    [0, 0, 1, 1, 0]
]


# ------------------------------------------------------------
# Archivo .txt
# ------------------------------------------------------------

nombre_archivo = "grafo.txt"


# ------------------------------------------------------------
# Generación aleatoria
# ------------------------------------------------------------

numero_nodos = 10

# Probabilidad de que exista una conexión
probabilidad_conexion = 0.3

# Semilla para reproducir el mismo grafo
semilla = 42


# ------------------------------------------------------------
# Coloreado
# ------------------------------------------------------------

# Número máximo de colores disponibles
numero_colores = 4

In [ ]:
# ============================================================
# GENERACIÓN DE MATRIZ ALEATORIA
# ============================================================

def generar_matriz_aleatoria(
    numero_nodos,
    probabilidad_conexion=0.3,
    semilla=None
):
    """
    Genera una matriz de adyacencia simétrica.

    1 = existe conexión
    0 = no existe conexión

    La diagonal siempre será 0.
    """

    if numero_nodos >= 21:
        raise ValueError(
            "El número de nodos debe ser menor que 21."
        )

    if numero_nodos < 1:
        raise ValueError(
            "El número de nodos debe ser mayor que 0."
        )

    if semilla is not None:
        random.seed(semilla)

    matriz = [
        [0 for _ in range(numero_nodos)]
        for _ in range(numero_nodos)
    ]

    for i in range(numero_nodos):

        for j in range(i + 1, numero_nodos):

            if random.random() < probabilidad_conexion:

                matriz[i][j] = 1
                matriz[j][i] = 1

    return matriz

In [ ]:
# ============================================================
# LECTURA DE MATRIZ DESDE ARCHIVO
# ============================================================

def leer_matriz_archivo(nombre_archivo):
    """
    Lee una matriz de adyacencia desde un archivo .txt.

    Los valores pueden estar separados por espacios.
    """

    if not os.path.exists(nombre_archivo):
        raise FileNotFoundError(
            f"No se encontró el archivo: {nombre_archivo}"
        )

    matriz = []

    with open(nombre_archivo, "r", encoding="utf-8") as archivo:

        for linea in archivo:

            linea = linea.strip()

            if not linea:
                continue

            fila = [
                int(valor)
                for valor in linea.split()
            ]

            matriz.append(fila)

    return matriz

In [ ]:
# ============================================================
# VALIDACIÓN DE MATRIZ
# ============================================================

def validar_matriz(matriz):
    """
    Verifica que la matriz sea:
    - cuadrada
    - simétrica
    - tenga únicamente 0 y 1
    - tenga ceros en la diagonal
    - N < 21
    """

    n = len(matriz)

    if n == 0:
        raise ValueError(
            "La matriz no puede estar vacía."
        )

    if n >= 21:
        raise ValueError(
            "El número de nodos debe ser menor que 21."
        )

    # Verificar que sea cuadrada
    for fila in matriz:

        if len(fila) != n:
            raise ValueError(
                "La matriz debe ser cuadrada (NxN)."
            )

    # Verificar valores
    for i in range(n):

        for j in range(n):

            if matriz[i][j] not in (0, 1):
                raise ValueError(
                    "La matriz solo puede contener 0 y 1."
                )

    # Verificar diagonal
    for i in range(n):

        if matriz[i][i] != 0:
            raise ValueError(
                "La diagonal principal debe contener únicamente 0."
            )

    # Verificar simetría
    for i in range(n):

        for j in range(n):

            if matriz[i][j] != matriz[j][i]:

                raise ValueError(
                    "La matriz debe ser simétrica."
                )

    return True

In [ ]:
# ============================================================
# OBTENER MATRIZ
# ============================================================

if fuente_matriz == "manual":

    matriz_adyacencia = matriz

elif fuente_matriz == "archivo":

    matriz_adyacencia = leer_matriz_archivo(
        nombre_archivo
    )

elif fuente_matriz == "aleatoria":

    matriz_adyacencia = generar_matriz_aleatoria(
        numero_nodos,
        probabilidad_conexion,
        semilla
    )

else:

    raise ValueError(
        "La fuente debe ser: "
        "'manual', 'archivo' o 'aleatoria'."
    )


validar_matriz(matriz_adyacencia)


print("Matriz de adyacencia:")
for fila in matriz_adyacencia:
    print(fila)

print()
print(f"Número de nodos: {len(matriz_adyacencia)}")

In [ ]:
# ============================================================
# CREAR GRAFO
# ============================================================

def matriz_a_grafo(matriz):
    """
    Convierte una matriz de adyacencia en un grafo NetworkX.
    """

    n = len(matriz)

    G = nx.Graph()

    # Agregar nodos
    for i in range(n):
        G.add_node(i)

    # Agregar conexiones
    for i in range(n):

        for j in range(i + 1, n):

            if matriz[i][j] == 1:
                G.add_edge(i, j)

    return G


G = matriz_a_grafo(matriz_adyacencia)

print(
    f"Grafo creado: "
    f"{G.number_of_nodes()} nodos, "
    f"{G.number_of_edges()} conexiones."
)

In [ ]:
# ============================================================
# DOMINIOS
# ============================================================

def crear_dominios(numero_nodos, numero_colores):
    """
    Crea el dominio inicial de cada nodo.
    """

    colores = list(range(numero_colores))

    dominios = {
        nodo: colores.copy()
        for nodo in range(numero_nodos)
    }

    return dominios


dominios_iniciales = crear_dominios(
    len(matriz_adyacencia),
    numero_colores
)

print("Dominios iniciales:")

for nodo, dominio in dominios_iniciales.items():
    print(f"Nodo {nodo}: {dominio}")

In [ ]:
# ============================================================
# FUNCIONES AUXILIARES
# ============================================================

def obtener_vecinos(nodo, matriz):
    """
    Obtiene los vecinos de un nodo.
    """

    return [
        j
        for j in range(len(matriz))
        if matriz[nodo][j] == 1
    ]


def asignacion_consistente(
    nodo,
    color,
    asignacion,
    matriz
):
    """
    Verifica que asignar un color a un nodo
    no genere conflicto con sus vecinos.
    """

    vecinos = obtener_vecinos(nodo, matriz)

    for vecino in vecinos:

        if vecino in asignacion:

            if asignacion[vecino] == color:
                return False

    return True

In [ ]:
# ============================================================
# FORWARD CHECKING
# ============================================================

def forward_checking(
    nodo,
    color,
    asignacion,
    dominios,
    matriz
):
    """
    Aplica Forward Checking después de asignar un color.

    Elimina el color asignado de los dominios
    de los vecinos no asignados.

    Devuelve:
        - nuevos dominios
        - True si no hay conflicto
        - False si algún dominio queda vacío
    """

    nuevos_dominios = {
        variable: dominio.copy()
        for variable, dominio in dominios.items()
    }

    nuevos_dominios[nodo] = [color]

    vecinos = obtener_vecinos(nodo, matriz)

    for vecino in vecinos:

        # Solo modificar nodos todavía no asignados
        if vecino not in asignacion:

            if color in nuevos_dominios[vecino]:

                nuevos_dominios[vecino].remove(color)

            # Si el dominio queda vacío,
            # se produjo un conflicto
            if len(nuevos_dominios[vecino]) == 0:

                return nuevos_dominios, False

    return nuevos_dominios, True

In [ ]:
# ============================================================
# DYNAMIC ORDERING
# ============================================================

def seleccionar_variable(
    asignacion,
    dominios,
    matriz
):
    """
    Selecciona dinámicamente el siguiente nodo.

    Criterio principal:
        MRV - Minimum Remaining Values

    Criterio secundario:
        Mayor grado.
    """

    no_asignados = [
        nodo
        for nodo in dominios
        if nodo not in asignacion
    ]

    if not no_asignados:
        return None

    mejor_nodo = None
    mejor_criterio = None

    for nodo in no_asignados:

        cantidad_valores = len(
            dominios[nodo]
        )

        grado = sum(
            matriz[nodo]
        )

        criterio = (
            cantidad_valores,
            -grado
        )

        if (
            mejor_criterio is None
            or criterio < mejor_criterio
        ):
            mejor_criterio = criterio
            mejor_nodo = nodo

    return mejor_nodo

In [ ]:
# ============================================================
# BT-FC-DO
# ============================================================

def bt_fc_do(
    asignacion,
    dominios,
    matriz,
    estadisticas=None
):
    """
    Backtracking + Forward Checking +
    Dynamic Ordering.

    Retorna una asignación válida o None.
    """

    # --------------------------------------------------------
    # Inicializar estadísticas
    # --------------------------------------------------------

    if estadisticas is None:

        estadisticas = {
            "nodos_visitados": 0,
            "retrocesos": 0,
            "podas_fc": 0
        }

    # --------------------------------------------------------
    # Condición de éxito
    # --------------------------------------------------------

    if len(asignacion) == len(matriz):

        return asignacion.copy()

    # --------------------------------------------------------
    # Dynamic Ordering
    # --------------------------------------------------------

    nodo = seleccionar_variable(
        asignacion,
        dominios,
        matriz
    )

    if nodo is None:
        return asignacion.copy()

    estadisticas["nodos_visitados"] += 1

    # --------------------------------------------------------
    # Probar colores
    # --------------------------------------------------------

    for color in dominios[nodo]:

        # Verificar consistencia
        if not asignacion_consistente(
            nodo,
            color,
            asignacion,
            matriz
        ):
            continue

        # Crear nueva asignación
        nueva_asignacion = asignacion.copy()

        nueva_asignacion[nodo] = color

        # ----------------------------------------------------
        # Forward Checking
        # ----------------------------------------------------

        nuevos_dominios, valido = forward_checking(
            nodo,
            color,
            nueva_asignacion,
            dominios,
            matriz
        )

        if not valido:

            estadisticas["podas_fc"] += 1

            continue

        # ----------------------------------------------------
        # Llamada recursiva
        # ----------------------------------------------------

        resultado = bt_fc_do(
            nueva_asignacion,
            nuevos_dominios,
            matriz,
            estadisticas
        )

        if resultado is not None:

            return resultado

    # --------------------------------------------------------
    # Ningún color funcionó
    # --------------------------------------------------------

    estadisticas["retrocesos"] += 1

    return None

In [ ]:
# ============================================================
# EJECUTAR BT-FC-DO
# ============================================================

dominios = crear_dominios(
    len(matriz_adyacencia),
    numero_colores
)

asignacion_inicial = {}

estadisticas = {
    "nodos_visitados": 0,
    "retrocesos": 0,
    "podas_fc": 0
}


solucion = bt_fc_do(
    asignacion_inicial,
    dominios,
    matriz_adyacencia,
    estadisticas
)


print("=" * 60)
print("RESULTADO BT-FC-DO")
print("=" * 60)

if solucion is None:

    print(
        f"No existe una solución utilizando "
        f"{numero_colores} colores."
    )

else:

    print("¡Solución encontrada!\n")

    for nodo in sorted(solucion):

        print(
            f"Nodo {nodo}: "
            f"Color {solucion[nodo]}"
        )

print()
print("Estadísticas:")
print(
    f"Nodos visitados: "
    f"{estadisticas['nodos_visitados']}"
)

print(
    f"Retrocesos: "
    f"{estadisticas['retrocesos']}"
)

print(
    f"Podas por Forward Checking: "
    f"{estadisticas['podas_fc']}"
)

In [ ]:
# ============================================================
# VALIDAR SOLUCIÓN
# ============================================================

def validar_coloreado(
    asignacion,
    matriz
):
    """
    Verifica que ningún par de nodos conectados
    tenga el mismo color.
    """

    if asignacion is None:
        return False

    n = len(matriz)

    for i in range(n):

        for j in range(i + 1, n):

            if matriz[i][j] == 1:

                if asignacion[i] == asignacion[j]:

                    return False

    return True


if solucion is not None:

    es_valida = validar_coloreado(
        solucion,
        matriz_adyacencia
    )

    print(
        "¿La solución es válida?:",
        "Sí" if es_valida else "No"
    )

else:

    print(
        "No hay solución que validar."
    )

In [ ]:
# ============================================================
# POSICIONES DEL GRAFO
# ============================================================

posiciones = nx.spring_layout(
    G,
    seed=42
)

In [ ]:
# ============================================================
# VISUALIZACIÓN DEL GRAFO COLOREADO
# ============================================================

plt.figure(figsize=(12, 9))

if solucion is not None:

    colores_nodos = [
        solucion[nodo]
        for nodo in G.nodes()
    ]

    nx.draw_networkx(
        G,
        posiciones,
        node_color=colores_nodos,
        cmap=plt.cm.tab10,
        node_size=1000,
        font_size=12,
        font_weight="bold",
        edge_color="gray",
        width=2
    )

    plt.title(
        f"Coloreado del grafo mediante BT-FC-DO\n"
        f"{numero_colores} colores",
        fontsize=16
    )

else:

    nx.draw_networkx(
        G,
        posiciones,
        node_size=1000,
        font_size=12,
        font_weight="bold",
        edge_color="gray",
        width=2
    )

    plt.title(
        "No existe una solución con "
        f"{numero_colores} colores",
        fontsize=16
    )


plt.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# RESUMEN FINAL
# ============================================================

print()
print("=" * 60)
print("RESUMEN DEL PROBLEMA")
print("=" * 60)

print(
    f"Número de nodos: "
    f"{len(matriz_adyacencia)}"
)

print(
    f"Número de conexiones: "
    f"{G.number_of_edges()}"
)

print(
    f"Colores disponibles: "
    f"{numero_colores}"
)

if solucion is not None:

    print("Estado: SOLUCIÓN ENCONTRADA")

    print("\nAsignación final:")

    for nodo in sorted(solucion):

        print(
            f"  Nodo {nodo} → "
            f"Color {solucion[nodo]}"
        )

else:

    print("Estado: SIN SOLUCIÓN")

print()
print(
    f"Nodos visitados: "
    f"{estadisticas['nodos_visitados']}"
)

print(
    f"Retrocesos: "
    f"{estadisticas['retrocesos']}"
)

print(
    f"Podas FC: "
    f"{estadisticas['podas_fc']}"
)